# Triage Copilot — app

Loads the trained model bundles from `models/` and serves them; it **trains nothing**. Two tabs: a triage screen for the agent (classification, routing, risk, suggested tags and similar past cases) and an exploration screen for inspecting each model.

## 1. Load the trained bundles

In [ ]:
import os, sys, warnings, html
warnings.filterwarnings("ignore")
os.environ["GRADIO_ANALYTICS_ENABLED"] = "False"      # no telemetry network calls
import numpy as np, pandas as pd, joblib

# Find and import the shared pipeline module (for retrieval / tag-suggestion helpers).
search = os.getcwd()
while search != os.path.dirname(search) and not os.path.exists(os.path.join(search, "case_pipeline.py")):
    search = os.path.dirname(search)
sys.path.insert(0, search)
import case_pipeline as P

def _load(key):
    path = os.path.join(P.MODELS_DIR, f"{key}.joblib")
    if not os.path.exists(path):
        raise FileNotFoundError(f"{path} missing — run training/train_{key}.ipynb first")
    return joblib.load(path)

# The five predictive models shown on the triage card (two of them for priority).
CLASSIFIERS = ["m1_case_type", "m2_category", "m3_priority_direct",
               "m3_priority_review", "m4_escalation", "m5_route"]
# Order used in the exploration tab (classifiers grouped sensibly).
EXPLORE_ORDER = ["m1_case_type", "m2_category", "m3_priority_direct",
                 "m3_priority_review", "m5_route", "m4_escalation"]

MODEL = {key: _load(key) for key in CLASSIFIERS}
RETRIEVER = _load("similar_cases")["retriever"]   # TF-IDF + nearest-neighbour index
TAGGER    = _load("tag_suggester")                # multi-label tag suggester (not a model input)
print("loaded", len(MODEL), "models + similar-case search + tag suggester")

## 2. Prediction cascade

Case type and category are predicted from the summary first, then feed the priority and escalation models. Tags are **never** an input — they are only suggested to the agent (section 3).

In [ ]:
PRIORITIES = MODEL["m3_priority_review"]["meta"]["priorities"]
PRIORITY_RANK = {level: i for i, level in enumerate(PRIORITIES)}

def _frame(base, extra=None):
    """Build a one-row DataFrame of features for a model."""
    return pd.DataFrame([{**base, **(extra or {})}])

def _ranked(key, row):
    """Classes of model `key` sorted by probability: [(label, prob), ...]."""
    bundle = MODEL[key]; model = bundle["model"]
    proba = model.predict_proba(row[bundle["features"]])[0]
    order = proba.argsort()[::-1]
    return [(str(model.classes_[i]), float(proba[i])) for i in order]

def predict_case(summary, channel, plan, month, tenure, agent_priority=None):
    # Shared inbound fields. No tags here — tags are leakage.
    base = {"case_summary": summary or "", "channel": channel, "plan_tier": plan,
            "month_num": int(month), "customer_tenure_months": int(tenure)}

    case_type = _ranked("m1_case_type", _frame(base))[0]
    category  = _ranked("m2_category",  _frame(base))[0]
    priority_direct = _ranked("m3_priority_direct", _frame(base))[0]                       # uses plan/tenure
    priority_review = _ranked("m3_priority_review", _frame(base, {"case_type": case_type[0]}))[0]
    route = _ranked("m5_route", _frame(base))[0]

    # Escalation uses the predicted category (available at triage via M2).
    esc_bundle = MODEL["m4_escalation"]; esc_model = esc_bundle["model"]
    esc_proba = esc_model.predict_proba(_frame(base, {"category": category[0]})[esc_bundle["features"]])[0]
    esc_prob = float(esc_proba[list(esc_model.classes_).index(True)])

    result = {
        "case_type": case_type, "category": category, "route": route,
        "priority_direct": priority_direct, "priority_review": priority_review,
        "priorities_agree": priority_direct[0] == priority_review[0],
        "esc_prob": esc_prob, "esc_flag": esc_prob >= esc_bundle["threshold"],
        "needs_review": route[1] < MODEL["m5_route"].get("review_threshold", 0.0),
        "terms": _top_terms("m1_case_type", _frame(base), 3),
    }
    # Optional priority double-check against the agent's own choice.
    if agent_priority in PRIORITY_RANK:
        gap = abs(PRIORITY_RANK[agent_priority] - PRIORITY_RANK[priority_review[0]])
        result["agent_priority"] = agent_priority
        result["priority_flag"] = gap >= MODEL["m3_priority_review"]["meta"]["flag_gap"]
    return result

# Words to drop from the plain-language "Why" explanation.
_STOPWORDS = set("the a an and or of to in for with on at is are be was were this that it its my our "
                 "your i we you they he she as by from not no cannot can keeps after getting get got need "
                 "want please thanks cheers hi hello there their them has have had do does did been being but "
                 "so if then when which who what why how able keep still just also very more most".split())

def _top_terms(key, row, k=3):
    """The top content words from the summary that drove model `key`'s prediction."""
    bundle = MODEL[key]; model = bundle["model"]
    features = model.named_steps["features"]; clf = model.named_steps["model"]
    names = features.get_feature_names_out()
    weights = np.asarray(features.transform(row[bundle["features"]]).todense()).ravel()
    class_idx = list(clf.classes_).index(model.predict(row[bundle["features"]])[0])
    coef = clf.coef_
    coef_row = coef[class_idx] if coef.shape[0] > 1 else (coef[0] if class_idx == 1 else -coef[0])
    contribution = coef_row * weights
    prefix = "case_summary_w__"                       # word-TF-IDF block only
    candidates = [i for i in range(len(names)) if names[i].startswith(prefix)]
    candidates.sort(key=lambda i: contribution[i], reverse=True)
    terms = []
    for i in candidates:
        if contribution[i] <= 0: break
        term = " ".join(w for w in names[i].replace(prefix, "").split()
                        if w not in _STOPWORDS and len(w) > 2)
        if term and term not in terms: terms.append(term)
        if len(terms) >= k: break
    return terms

print(predict_case("Cannot connect the Xero integration, OAuth token fails with a 500 error",
                    "in_app", "Enterprise", 6, 30))

## 3. Triage-tab renderers (card, suggested tags, similar cases)

In [ ]:
ESC_THRESHOLD = MODEL["m4_escalation"]["threshold"]

def _risk_band(prob):
    if prob >= ESC_THRESHOLD:     return "High",   "#dc2626", "#fef2f2"
    if prob >= ESC_THRESHOLD / 2: return "Medium", "#d97706", "#fffbeb"
    return "Low", "#059669", "#ecfdf5"

def _why(result):
    parts = []
    if result["terms"]:
        parts.append(", ".join(result["terms"]) + " in summary")
    parts.append(f'{result["case_type"][0].replace("_", " ")} language')
    if result["esc_flag"]:
        parts.append("flagged above the escalation threshold")
    elif result["esc_prob"] >= ESC_THRESHOLD / 2:
        parts.append("elevated but below the escalation threshold")
    else:
        parts.append("below the escalation threshold")
    return " + ".join(parts)

def render_card(r):
    level, colour, bg = _risk_band(r["esc_prob"])
    agree = "" if r["priorities_agree"] else ' <span class="tag warn">models differ</span>'
    rows = [
        ("Case type",        f'{r["case_type"][0].replace("_"," ").title()} <span class="muted">{r["case_type"][1]:.0%}</span>'),
        ("Category",         f'{r["category"][0].replace("_"," ").title()} <span class="muted">{r["category"][1]:.0%}</span>'),
        ("Priority · direct",f'{r["priority_direct"][0]} <span class="muted">{r["priority_direct"][1]:.0%}</span>'),
        ("Priority · review",f'{r["priority_review"][0]} <span class="muted">{r["priority_review"][1]:.0%}</span>{agree}'),
        ("Suggested route",  r["route"][0].title() + (' <span class="tag warn">needs human review</span>' if r["needs_review"] else "")),
    ]
    body = "".join(f'<tr><td class="k">{k}</td><td class="v">{v}</td></tr>' for k, v in rows)
    risk = (f'<tr><td class="k">Risk</td><td class="v"><span class="pill" '
            f'style="color:{colour};background:{bg}">{level} escalation risk · {r["esc_prob"]:.0%}</span></td></tr>')
    flag = ""
    if r.get("priority_flag"):
        flag = (f'<tr><td class="k">Priority check</td><td class="v"><span class="pill" '
                f'style="color:#dc2626;background:#fef2f2">review: you set {r["agent_priority"]}, '
                f'estimate is {r["priority_review"][0]}</span></td></tr>')
    why = f'<tr><td class="k">Why</td><td class="v why">{html.escape(_why(r))}</td></tr>'
    return f'<div class="rescard"><table class="res">{body}{risk}{flag}{why}</table></div>'

def render_tags(summary):
    """Suggested tags — a workflow aid. These are NOT used as model inputs."""
    tags = P.suggest_tags(TAGGER, summary, k=5)
    if not tags:
        chips = '<span class="muted">no confident tag suggestions</span>'
    else:
        chips = "".join(f'<span class="chip-tag">{html.escape(t)} <span class="muted">{p:.0%}</span></span>' for t, p in tags)
    return (f'<div class="rescard"><div class="sec">Suggested tags</div>{chips}'
            f'<p class="note">A workflow aid for the agent — the models do not use tags as inputs (tags are leakage).</p></div>')

def render_similar(summary, k=3):
    """Most similar historical cases and how they were resolved."""
    sim = P.query_similar(RETRIEVER, summary, k=k)
    items = ""
    for _, c in sim.iterrows():
        rt = c.get("resolution_time_hours")
        rt = f'{rt:.0f}h' if pd.notna(rt) else "—"
        items += (f'<div class="simrow"><div class="simtop">'
                  f'<span class="simsum">{html.escape(str(c["case_summary"])[:90])}</span>'
                  f'<span class="simscore">{c["similarity"]:.0%}</span></div>'
                  f'<div class="simmeta">team <b>{html.escape(str(c["assigned_team"]))}</b> · '
                  f'resolution <b>{html.escape(str(c.get("resolution_code") or "—"))}</b> · '
                  f'{rt} · {html.escape(str(c.get("status") or "—"))}</div></div>')
    return (f'<div class="rescard"><div class="sec">Similar past cases — how they were resolved</div>{items}</div>')

print(render_card(predict_case("Dashboard is down for all users, 500 errors everywhere","phone","Pro",9,36))[:60], "...")

## 4. Exploration-tab helpers

In [ ]:
def model_cards():
    cards = ""
    for key in EXPLORE_ORDER:
        b = MODEL[key]; meta = b["meta"]; m = b["metrics"]
        core = ["accuracy", "macro_F1", "weighted_F1", "cohen_kappa", "ROC_AUC", "PR_AUC"]
        core_str = " · ".join(f'{c} <b>{m[c]}</b>' for c in core if c in m)
        extra_keys = [k for k in m if k not in core + ["n_test", "baseline_acc"]]
        extra_str = " · ".join(f'{k} <b>{m[k]}</b>' for k in extra_keys)
        cards += (f'<div class="mcard"><div class="mh">{key.split("_")[0].upper()} — {b["label"]}'
                  f'<span class="chip">{meta.get("contributor","")}</span></div><div class="mb">'
                  f'<div class="spec"><span>Approach</span>{meta["approach"]}</div>'
                  f'<div class="spec"><span>Inputs</span>{meta["inputs"]}</div>'
                  f'<div class="spec"><span>Split</span>{meta["split"]}</div>'
                  f'<div class="spec"><span>Core</span>{core_str}</div>'
                  + (f'<div class="spec"><span>Signature</span>{extra_str}</div>' if extra_str else "")
                  + (f'<div class="cav">{meta.get("note","")}</div>' if meta.get("note") else "")
                  + "</div></div>")
    return f'<div class="mgrid">{cards}</div>'

def metrics_table():
    rows = []
    for key in EXPLORE_ORDER:
        b = MODEL[key]; row = {"Model": key.split("_")[0].upper(), "Predicts": b["label"]}
        row.update(b["metrics"]); rows.append(row)
    cols = ["Model","Predicts","n_test","accuracy","macro_F1","weighted_F1","cohen_kappa","ROC_AUC","PR_AUC","baseline_acc"]
    frame = pd.DataFrame(rows)
    return frame[[c for c in cols if c in frame.columns]]

def case_choices(key):
    test = MODEL[key]["test"]
    return [(f'{r.case_id} — {str(r.case_summary)[:55]}', i) for i, r in test.iterrows()]

def inspect(key, idx):
    if key is None or idx is None:
        return '<div class="rescard"><p class="note">Pick a model and a case.</p></div>'
    b = MODEL[key]; row = b["test"].iloc[int(idx)]
    actual, pred = row[b["target"]], row["predicted"]; conf = row.get("confidence")
    ok = str(pred) == str(actual)
    mark = '<span class="tag ok">✓ match</span>' if ok else '<span class="tag warn">✗ differs</span>'
    extra = f' · prob {row["prob"]:.0%} vs threshold {b.get("threshold",0):.0%}' if b["kind"] == "binary" else ""
    return (f'<div class="rescard"><p class="sum">“{html.escape(str(row.case_summary)[:300])}”</p>'
            f'<table class="insp"><tr><td class="k">Model</td><td>{key.split("_")[0].upper()} — {b["label"]}</td></tr>'
            f'<tr><td class="k">Predicted</td><td>{html.escape(str(pred))}' + (f' ({conf:.0%})' if conf is not None else "") + f'{extra}</td></tr>'
            f'<tr><td class="k">Actual</td><td>{html.escape(str(actual))}</td></tr>'
            f'<tr><td class="k">Result</td><td>{mark}</td></tr></table>'
            f'<p class="note">Held-out case ({b["meta"]["split"]} split) — never seen in training.</p></div>')

def confusion_html(key):
    cm = MODEL[key]["confusion"]; labels = cm["labels"]; matrix = cm["matrix"]
    peak = max((max(r) for r in matrix), default=1) or 1
    head = "".join(f'<th>{html.escape(str(l))[:10]}</th>' for l in labels)
    body = ""
    for i, label in enumerate(labels):
        cells_ = ""
        for j in range(len(labels)):
            v = matrix[i][j]; shade = 0.08 + 0.72 * (v / peak)
            bg = f"rgba(79,70,229,{shade:.2f})" if v else "#fff"
            cells_ += f'<td style="background:{bg}">{v}</td>'
        body += f'<tr><th>{html.escape(str(label))[:10]}</th>{cells_}</tr>'
    return (f'<div class="rescard"><div class="small muted">rows = actual · columns = predicted</div>'
            f'<table class="cm"><tr><th></th>{head}</tr>{body}</table></div>')

def report_html(key):
    report = MODEL[key]["report"]; rows = ""
    for cls, d in report.items():
        if cls in ("accuracy", "macro avg", "weighted avg") or not isinstance(d, dict): continue
        rows += (f'<tr><td>{html.escape(str(cls))}</td><td>{d["precision"]:.2f}</td>'
                 f'<td>{d["recall"]:.2f}</td><td>{d["f1-score"]:.2f}</td><td>{int(d["support"])}</td></tr>')
    return f'<div class="rescard"><table class="rep"><tr><th>class</th><th>prec</th><th>rec</th><th>F1</th><th>n</th></tr>{rows}</table></div>'

print("exploration helpers ready |", metrics_table().shape)

## 5. App — modern two-tab UI

In [ ]:
import gradio as gr

CHANNELS = ["email", "webchat", "in_app", "phone"]
PLANS    = ["Free", "Standard", "Pro", "Enterprise"]
EXPLORE_LABELS = {k: f'{k.split("_")[0].upper()} — {MODEL[k]["label"]}' for k in EXPLORE_ORDER}
NOTEBOOK_PATH  = {k: os.path.join(P.ROOT, "training", f"train_{k}.ipynb") for k in EXPLORE_ORDER}
EXAMPLE = ["Cannot connect the Xero integration — the OAuth token keeps failing with a 500 error after the latest update",
           "in_app", "Enterprise", 6, 30, "(none)"]

CSS = """
:root{--ink:#0f172a;--muted:#64748b;--line:#e2e8f0;--accent:#4f46e5;--bg:#f8fafc;}
.gradio-container{background:var(--bg);}
#wrap{max-width:1120px;margin:auto;}
.hero{background:linear-gradient(120deg,#4f46e5,#0ea5e9);color:#fff;border-radius:18px;padding:22px 26px;margin-bottom:14px;}
.hero h1{margin:0;font-size:24px;font-weight:800;letter-spacing:-.3px;}
.hero p{margin:6px 0 0;opacity:.92;font-size:14px;}
.rescard{border:1px solid var(--line);border-radius:16px;padding:18px 22px;background:#fff;box-shadow:0 1px 2px rgba(15,23,42,.04);margin-bottom:12px;}
.sec{font-size:12px;font-weight:700;text-transform:uppercase;letter-spacing:.04em;color:var(--muted);margin-bottom:10px;}
table.res{width:100%;border-collapse:collapse;}
table.res td{padding:11px 6px;border-bottom:1px solid #f1f5f9;font-size:16px;vertical-align:top;}
table.res tr:last-child td{border-bottom:none;}
td.k{color:var(--muted);width:150px;font-weight:600;font-size:12px;text-transform:uppercase;letter-spacing:.03em;}
td.v{color:var(--ink);} td.v.why{color:#334155;font-size:15px;} .muted{color:var(--muted);font-weight:500;font-size:14px;}
.pill{padding:5px 13px;border-radius:999px;font-weight:700;font-size:14px;}
.tag{padding:2px 9px;border-radius:999px;font-size:12px;font-weight:700;}
.tag.warn{color:#b45309;background:#fffbeb;} .tag.ok{color:#047857;background:#ecfdf5;}
.chip-tag{display:inline-block;background:#eef2ff;color:#4338ca;border-radius:999px;padding:4px 11px;margin:0 6px 6px 0;font-size:13px;font-weight:600;}
.simrow{border-top:1px solid #f1f5f9;padding:9px 0;} .simrow:first-of-type{border-top:none;}
.simtop{display:flex;justify-content:space-between;gap:10px;} .simsum{font-size:14px;color:var(--ink);}
.simscore{color:var(--accent);font-weight:700;font-size:13px;} .simmeta{color:var(--muted);font-size:12px;margin-top:2px;}
.sum{font-style:italic;color:#334155;margin:0 0 14px;} .note{color:#94a3b8;font-size:12px;margin-top:10px;}
.small{font-size:12px;margin-bottom:6px;}
.mgrid{display:grid;grid-template-columns:repeat(2,1fr);gap:14px;}
.mcard{border:1px solid var(--line);border-radius:14px;overflow:hidden;background:#fff;}
.mh{background:#0f172a;color:#fff;padding:11px 14px;font-weight:700;font-size:14px;display:flex;justify-content:space-between;align-items:center;}
.chip{background:rgba(255,255,255,.16);border-radius:999px;padding:1px 9px;font-size:11px;font-weight:700;}
.mb{padding:13px 15px;font-size:13px;color:var(--ink);line-height:1.5;}
.spec{margin-bottom:6px;} .spec>span{display:inline-block;width:84px;color:var(--muted);font-weight:700;vertical-align:top;}
.cav{color:#92400e;font-size:12px;margin-top:8px;padding-top:8px;border-top:1px dashed var(--line);}
table.insp{width:100%;border-collapse:collapse;} table.insp td{padding:8px 6px;border-bottom:1px solid #f1f5f9;font-size:14px;}
table.cm{border-collapse:collapse;font-size:12px;} table.cm th{color:var(--muted);padding:4px 7px;font-weight:700;}
table.cm td{padding:6px 9px;text-align:center;border:1px solid #f1f5f9;min-width:34px;}
table.rep{width:100%;border-collapse:collapse;font-size:13px;} table.rep th{text-align:left;color:var(--muted);border-bottom:2px solid var(--line);padding:5px;}
table.rep td{padding:5px;border-bottom:1px solid #f1f5f9;}
"""

with gr.Blocks(title="Triage Copilot", css=CSS, theme=gr.themes.Soft(primary_hue="indigo")) as demo:
    with gr.Column(elem_id="wrap"):
        gr.HTML('<div class="hero"><h1>Northstar Triage Copilot</h1>'
                '<p>Classify, route, prioritise and flag risk for a new case — with suggested tags and similar past cases.</p></div>')
        with gr.Tabs() as tabs:
            # ---------- Triage ----------
            with gr.Tab("Triage a case", id="triage"):
                with gr.Row():
                    with gr.Column(scale=5):
                        summary = gr.Textbox(label="Case summary", lines=4, placeholder="Paste the customer's case summary…")
                        with gr.Row():
                            channel = gr.Dropdown(CHANNELS, value="email", label="Channel")
                            plan = gr.Dropdown(PLANS, value="Standard", label="Plan tier (direct-priority model only)")
                        with gr.Row():
                            month = gr.Dropdown(list(range(1, 13)), value=6, label="Month")
                            tenure = gr.Number(value=12, label="Tenure (months)")
                        agent_pri = gr.Dropdown(["(none)"] + PRIORITIES, value="(none)",
                                                label="Agent's priority (optional — triggers the review check)")
                        with gr.Row():
                            go = gr.Button("Triage case", variant="primary", scale=2)
                            ex = gr.Button("Load example", scale=1)
                    with gr.Column(scale=5):
                        out_card = gr.HTML('<div class="rescard"><p class="note">Enter a case and press <b>Triage case</b>, or <b>Load example</b>.</p></div>')
                        out_tags = gr.HTML()
                        out_similar = gr.HTML()
                        explore_btn = gr.Button("Explore how the models work →", variant="secondary")

                def triage(summary, channel, plan, month, tenure, agent_priority):
                    result = predict_case(summary, channel, plan, month, tenure,
                                          None if agent_priority == "(none)" else agent_priority)
                    return render_card(result), render_tags(summary), render_similar(summary)

                inputs = [summary, channel, plan, month, tenure, agent_pri]
                outputs = [out_card, out_tags, out_similar]
                go.click(triage, inputs, outputs)
                ex.click(lambda: EXAMPLE, None, inputs).then(triage, inputs, outputs)

            # ---------- Exploration ----------
            with gr.Tab("Explore the models", id="explore"):
                gr.Markdown("### Model cards")
                gr.HTML(model_cards())
                gr.Markdown("### Normalised held-out metrics")
                gr.Dataframe(metrics_table(), interactive=False)
                gr.Markdown("### Inspect a held-out case — predicted vs actual")
                with gr.Row():
                    model_pick = gr.Dropdown([(EXPLORE_LABELS[k], k) for k in EXPLORE_ORDER], label="Model")
                    case_pick = gr.Dropdown([], label="Held-out case")
                inspect_out = gr.HTML('<div class="rescard"><p class="note">Pick a model, then a case.</p></div>')
                model_pick.change(lambda k: gr.update(choices=case_choices(k), value=None), model_pick, case_pick)
                case_pick.change(inspect, [model_pick, case_pick], inspect_out)
                gr.Markdown("### Confusion matrix & per-class report")
                model_pick2 = gr.Dropdown([(EXPLORE_LABELS[k], k) for k in EXPLORE_ORDER], value="m1_case_type", label="Model")
                with gr.Row():
                    cm_out = gr.HTML(confusion_html("m1_case_type"))
                    rep_out = gr.HTML(report_html("m1_case_type"))
                model_pick2.change(lambda k: (confusion_html(k), report_html(k)), model_pick2, [cm_out, rep_out])
                gr.Markdown("### Training notebooks")
                with gr.Row():
                    for k in EXPLORE_ORDER:
                        gr.DownloadButton(f"⬇ {k}", value=NOTEBOOK_PATH[k], size="sm")

        # Button on the triage tab jumps to the exploration tab.
        explore_btn.click(lambda: gr.Tabs(selected="explore"), None, tabs)

print("app built | models:", len(MODEL))

## 6. Launch

In [ ]:
# demo.launch(server_name="127.0.0.1", server_port=7866, inbrowser=False)